In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import numpy as np
from src.data import load_region_df
from src.soc_engine import BatterySpec
from src.forecaster import (
    BATTERY_LZ_COL,
    train_houston_price_model,
    generate_forecast_range,
)
from src.dam_bidder import run_dam_campaign, dam_campaign_kpis
from src.forecast_error import (
    run_perfect_foresight,
    run_perfect_foresight_campaign,
    compute_forecast_error_analysis,
    forecast_error_summary,
    monthly_error_breakdown,
    capture_ratio_distribution,
)

passed = 0
failed = 0

def check(condition, msg_pass, msg_fail):
    global passed, failed
    if condition:
        print(f"✅ {msg_pass}")
        passed += 1
    else:
        print(f"❌ {msg_fail}")
        failed += 1

# ── Setup ──────────────────────────────────────────────────────────────────
df_coast   = load_region_df("coast")
spec       = BatterySpec(power_mw=100, duration_hours=4)
test_start = pd.Timestamp("2025-01-01")

test_day      = pd.Timestamp("2025-06-15")
actual_day    = df_coast[df_coast.index.date == test_day.date()]
actual_prices = actual_day[BATTERY_LZ_COL]

# ── Test 1 — run_perfect_foresight returns 24 rows ────────────────────────
pf_day = run_perfect_foresight(actual_prices, spec)

check(len(pf_day) == 24,
      "Test 1 — perfect foresight returns 24 rows",
      f"Test 1 — wrong row count: {len(pf_day)}")
check("net_revenue" in pf_day.columns,
      "Test 1 — net_revenue column present",
      "Test 1 — net_revenue MISSING")
check("soc_pct" in pf_day.columns,
      "Test 1 — soc_pct column present",
      "Test 1 — soc_pct MISSING")

# ── Test 2 — PF revenue >= 0 (never dispatches unprofitably) ─────────────
pf_total = pf_day["net_revenue"].sum()
check(
    pf_total >= 0,
    f"Test 2 — PF revenue >= 0: ${pf_total:,.2f}",
    f"Test 2 — PF dispatched unprofitably: ${pf_total:,.2f}"
)

# ── Test 3 — PF discharge always after charge (causal) ────────────────────
charge_hours    = pf_day[pf_day["charge_mw"]    > 0].index
discharge_hours = pf_day[pf_day["discharge_mw"] > 0].index

if len(charge_hours) > 0 and len(discharge_hours) > 0:
    check(
        discharge_hours.min() > charge_hours.max(),
        f"Test 3 — discharge after charge: charge ends {charge_hours.max().hour}:00, discharge starts {discharge_hours.min().hour}:00",
        "Test 3 — causal constraint violated in PF"
    )
else:
    check(True, "Test 3 — battery idle (no profitable opportunity) — causal N/A", "")

In [ ]:
# ── Test 4 — PF >= DAM revenue on the same day ────────────────────────────
print("\nTraining model and generating forecast for comparison...")
model_result = train_houston_price_model(df_coast, test_start)
forecast_df  = generate_forecast_range(
    model_result, df_coast,
    pd.Timestamp("2025-06-15"),
    pd.Timestamp("2025-06-15"),
)
dam_single = run_dam_campaign(forecast_df, df_coast, spec)

if len(dam_single) > 0:
    dam_revenue = dam_single["net_revenue"].sum()
    check(
        pf_total >= dam_revenue - 0.01,
        f"Test 4 — PF (${pf_total:,.2f}) >= DAM (${dam_revenue:,.2f}): "
        f"PF is always upper bound",
        f"Test 4 — DAM (${dam_revenue:,.2f}) > PF (${pf_total:,.2f}): "
        f"upper bound violated"
    )
else:
    check(pf_total >= 0, "Test 4 — no DAM days (single day); PF >= 0", "Test 4 — skip")

# ── Test 5 — run_perfect_foresight_campaign returns one row per day ────────
print("\nRunning PF campaign over test window...")
test_window = df_coast[df_coast.index >= test_start].copy()
pf_campaign = run_perfect_foresight_campaign(test_window, spec)

check(
    len(pf_campaign) > 0,
    f"Test 5 — PF campaign returned {len(pf_campaign)} days",
    "Test 5 — PF campaign returned 0 rows"
)
check(
    "pf_net_revenue" in pf_campaign.columns,
    "Test 5 — pf_net_revenue column present",
    "Test 5 — pf_net_revenue MISSING"
)
check(
    (pf_campaign["pf_net_revenue"] >= 0).all(),
    "Test 5 — all PF daily revenues >= 0",
    f"Test 5 — negative PF revenues found: {(pf_campaign['pf_net_revenue'] < 0).sum()} days"
)
print(f"  PF campaign: total=${pf_campaign['pf_net_revenue'].sum():,.0f} | avg daily=${pf_campaign['pf_net_revenue'].mean():,.2f}\n")

In [ ]:
# ── Test 6 — compute_forecast_error_analysis merges correctly ─────────────
full_forecast = generate_forecast_range(
    model_result, df_coast,
    pd.Timestamp(test_start),
    pd.Timestamp(df_coast.index.max().date()),
)
dam_campaign = run_dam_campaign(full_forecast, df_coast, spec)
analysis_df  = compute_forecast_error_analysis(pf_campaign, dam_campaign)

check(
    isinstance(analysis_df, pd.DataFrame),
    "Test 6 — analysis returns DataFrame",
    "Test 6 — wrong type"
)
required_cols = [
    "date", "pf_net_revenue", "dam_revenue",
    "forecast_error_cost", "capture_ratio", "idle_day"
]
for col in required_cols:
    check(col in analysis_df.columns,
          f"Test 6 — '{col}' column present",
          f"Test 6 — '{col}' MISSING")

# ── Test 7 — forecast_error_cost = pf - dam for every row ─────────────────
errors = (
    analysis_df["forecast_error_cost"] -
    (analysis_df["pf_net_revenue"] - analysis_df["dam_revenue"])
).abs()
check(
    (errors < 0.01).all(),
    "Test 7 — forecast_error_cost = pf - dam for every row",
    f"Test 7 — arithmetic error in {(errors >= 0.01).sum()} rows"
)

# ── Test 8 — PF revenue always >= DAM revenue ─────────────────────────────
violations = (
    analysis_df["dam_revenue"] > analysis_df["pf_net_revenue"] + 0.01
).sum()
check(
    violations == 0,
    "Test 8 — PF revenue >= DAM revenue on every day (upper bound holds)",
    f"Test 8 — upper bound violated on {violations} days"
)

# ── Test 9 — capture ratio bounds ─────────────────────────────────────────
above_100 = (analysis_df["capture_ratio"] > 100.5).sum()
check(
    above_100 == 0,
    "Test 9 — no capture ratio > 100% (PF is always upper bound)",
    f"Test 9 — {above_100} days with capture ratio > 100%"
)

In [ ]:
# ── Test 10 — forecast_error_summary structure ────────────────────────────
summary = forecast_error_summary(analysis_df, spec)
required_keys = [
    "total_pf_revenue", "total_dam_revenue",
    "total_forecast_error_cost", "overall_capture_ratio",
    "avg_daily_pf_revenue", "avg_daily_dam_revenue",
    "avg_daily_error_cost", "avg_capture_ratio",
    "p25_capture_ratio", "p75_capture_ratio",
    "worst_error_cost_day", "worst_error_cost",
    "days_capture_below_50pct", "idle_days",
    "pf_revenue_per_mw_year", "dam_revenue_per_mw_year",
    "error_cost_per_mw_year",
]
for key in required_keys:
    check(key in summary,
          f"Test 10 — '{key}' present in summary",
          f"Test 10 — '{key}' MISSING from summary")

check(
    summary["total_pf_revenue"] >= summary["total_dam_revenue"],
    f"Test 10 — total PF (${summary['total_pf_revenue']:,.0f}) >= total DAM (${summary['total_dam_revenue']:,.0f})",
    "Test 10 — total DAM exceeds total PF — impossible"
)
check(
    0 <= summary["overall_capture_ratio"] <= 100,
    f"Test 10 — overall capture ratio in range: {summary['overall_capture_ratio']:.1f}%",
    f"Test 10 — capture ratio out of range: {summary['overall_capture_ratio']}"
)

print(f"\n  Forecast Error Summary:")
print(f"    Perfect Foresight:    ${summary['total_pf_revenue']:>12,.0f}")
print(f"    Forecast-Driven DAM:  ${summary['total_dam_revenue']:>12,.0f}")
print(f"    Error Cost:           ${summary['total_forecast_error_cost']:>12,.0f}")
print(f"    Capture Ratio:         {summary['overall_capture_ratio']:>11.1f}%")
print(f"    Error Cost / MW / Yr: ${summary['error_cost_per_mw_year']:>12,.0f}\n")

# ── Test 11 — monthly_error_breakdown structure ────────────────────────────
monthly = monthly_error_breakdown(analysis_df)
check(isinstance(monthly, pd.DataFrame),
      "Test 11 — monthly breakdown returns DataFrame",
      "Test 11 — wrong type")
check("Month" in monthly.columns,
      "Test 11 — Month column present",
      "Test 11 — Month column MISSING")
check("Forecast Error Cost ($)" in monthly.columns,
      "Test 11 — Forecast Error Cost column present",
      "Test 11 — Forecast Error Cost MISSING")
check("Avg Capture Ratio (%)" in monthly.columns,
      "Test 11 — Avg Capture Ratio column present",
      "Test 11 — Avg Capture Ratio MISSING")

# ── Test 12 — capture_ratio_distribution sums to ~100% ────────────────────
dist = capture_ratio_distribution(analysis_df)
total_pct = (
    dist["pct_days_excellent"] +
    dist["pct_days_good"] +
    dist["pct_days_moderate"] +
    dist["pct_days_poor"] +
    dist["pct_days_negative"]
)
check(
    abs(total_pct - 100.0) < 1.0,
    f"Test 12 — regime percentages sum to ~100%: {total_pct:.1f}%",
    f"Test 12 — regime percentages sum to {total_pct:.1f}% — should be ~100%"
)
print(f"\n  Capture Ratio Distribution:")
print(f"    Excellent (>90%):  {dist['pct_days_excellent']:>5.1f}% of days")
print(f"    Good    (70-90%):  {dist['pct_days_good']:>5.1f}% of days")
print(f"    Moderate(50-70%):  {dist['pct_days_moderate']:>5.1f}% of days")
print(f"    Poor    (0-50%):   {dist['pct_days_poor']:>5.1f}% of days")
print(f"    Negative   (<0%):  {dist['pct_days_negative']:>5.1f}% of days")

# ── Test 13 — no Streamlit imports ────────────────────────────────────────
with open("../src/forecast_error.py") as f:
    src = f.read()
check(
    "import streamlit" not in src and "from streamlit" not in src,
    "Test 13 — no Streamlit imports in forecast_error.py",
    "Test 13 — Streamlit import found"
)

print(f"\n{'─'*50}")
print(f"Results: {passed} passed / {failed} failed / {passed+failed} total")
if failed == 0:
    print("🎉 All tests passed — forecast error analysis ready for Step 6")
else:
    print("⚠️  Fix failing tests before proceeding")
print(f"{'─'*50}")